In [ ]:
"""Environment setup — run this cell first before any imports."""
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"          # use GPU device 2; PyTorch sees it as cuda:0
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces fragmentation

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
if device == "cuda":
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free   = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {free:.1f} GB free / {total:.1f} GB total")

# Notebook 02 — Feature Analysis (DINO ViT-B/16)
**Owner: Person B — Week 1–2**

Stage 2. Retrieve top-activating patches from the 5,000-image ImageNet cache,
auto-label with CLIP, compute Monosemanticity Scores, and produce the feature catalog.

Outputs (top patches, MS scores, CLIP labels) are consumed by notebook 03 (CaFE locality analysis).

Requires: activation cache built in notebook 01.

## 1. Build / load activation cache

Run cell 1a once to extract the zip and collect image paths.
Run cell 1b once to build the HDF5 cache (~10–20 min on GPU for 5,000 images).
`build_cache` is guarded and will not overwrite an existing file.

In [ ]:
"""Step 1a — extract zip and collect image paths (run once)."""
import zipfile
from pathlib import Path
from src.config import get_config
import src.config as _cfg_mod

cfg = get_config()
repo_root  = _cfg_mod._DEFAULT_CONFIG.parent.parent
data_dir   = repo_root / cfg.data.imagenet_val_path
zip_path   = repo_root / "data" / "imagenet_val_5000.zip"
cache_path = repo_root / cfg.outputs.cache_path

flat_dir = data_dir
if zip_path.exists() and not (flat_dir.exists() and any(flat_dir.glob("*.jpg"))):
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(repo_root / "data")
    print(f"Extracted {sum(1 for _ in flat_dir.glob('*.jpg'))} images -> {flat_dir}")
else:
    n = sum(1 for _ in flat_dir.glob("*.jpg")) if flat_dir.exists() else 0
    print(f"Images already present: {n} found at {flat_dir}")

image_paths = sorted(str(p.resolve()) for p in flat_dir.glob("*.jpg"))
labels    = ["unknown"] * len(image_paths)
class_ids = [-1] * len(image_paths)

print(f"Total images: {len(image_paths)}")

In [ ]:
"""Step 1b — build cache (run once, ~10–20 min on MPS for 5000 images)."""
from src.cache import build_cache

if cache_path.exists():
    print(f"Cache already exists at {cache_path} — skipping build.")
    print("Delete the file and re-run if you want to rebuild.")
else:
    print(f"Building cache for {len(image_paths)} images, layers {cfg.sae.target_layers} ...")
    build_cache(image_paths, labels, class_ids, outputpath=str(cache_path))
    print(f"Done. Cache saved to {cache_path}")

In [ ]:
"""Step 1c — load activations for all target layers from cache."""
from src.cache import load_layer, load_image_index, load_metadata

meta  = load_metadata(cachepath=str(cache_path))
index = load_image_index(cachepath=str(cache_path))

print(f"Model     : {meta['model_name']}")
print(f"Layers    : {meta['layers']}")
print(f"Images    : {meta['n_images']}")
print(f"Image paths sample: {index['paths'][:2]}")

activations_by_layer = {}
for layer in cfg.sae.target_layers:
    activations_by_layer[layer] = load_layer(layer, cachepath=str(cache_path))
    print(f"  layer {layer}: {tuple(activations_by_layer[layer].shape)}")

# Primary-layer alias for the smoke test cell below
activations = activations_by_layer[cfg.sae.primary_layer]
print(f"\nactivations (primary layer {cfg.sae.primary_layer}): {tuple(activations.shape)}")

## 2. Top patches — single feature test

In [ ]:
"""Single-feature smoke test — check one feature before running all 49k."""
import importlib, src.features
importlib.reload(src.features)
from src.features import get_top_patches, crop_patch_images

# Try feature 0 first; swap for any index in [0, sae.d_sae)
top = get_top_patches(
    layer=cfg.sae.primary_layer,
    feature_idx=0,
    activations=activations,
    image_paths=index["paths"],
)
print(f"Top {len(top)} patches for feature 0:")
for p in top[:3]:
    print(f"  img {p['image_idx']:4d}  token {p['token_idx']:3d}"
          f"  row {p['patch_row']}  col {p['patch_col']}"
          f"  activation {p['activation_value']:.4f}")

# Visual sanity check — display top patches with red box marking the active patch
crops = crop_patch_images(top, context_patches=2, mark_patch=True)
if crops:
    from PIL import Image as _PIL_Image
    row_img = _PIL_Image.new("RGB", (128 * len(crops), 128), "white")
    for i, c in enumerate(crops):
        row_img.paste(c.resize((128, 128)), (i * 128, 0))
    display(row_img)
else:
    print("No patch images to display (check image_paths resolve correctly).")

## 3. Top patches — all features, all layers

In [ ]:
"""All features — streaming top-k for each target layer.
Peak GPU memory ≈ batch_size × 197 × 49152 × 4 bytes:
  batch_size=32  → ~1.2 GB  (safe when < 4 GB free)
  batch_size=64  → ~2.4 GB  (safe when < 8 GB free)   ← default
  batch_size=128 → ~4.8 GB  (safe when < 8 GB free)
  batch_size=256 → ~9.5 GB  (only if GPU is mostly empty)
Check free VRAM in cell 1 output before choosing.
"""
import gzip, pickle
from pathlib import Path
from src.features import get_top_patches_all_features

feature_output_dir = repo_root / "outputs" / "features"
feature_output_dir.mkdir(parents=True, exist_ok=True)

_name_to_local = {Path(p).name: p for p in image_paths}

all_top_patches_by_layer = {}

for layer in cfg.sae.target_layers:
    pkl_path = feature_output_dir / f"top_patches_layer{layer}_full.pkl.gz"

    if pkl_path.exists():
        with gzip.open(pkl_path, "rb") as _f:
            patches = pickle.load(_f)
        print(f"[cached] layer {layer}: {len(patches):,} features — {pkl_path.name}")
    else:
        print(f"Computing layer {layer} top patches …")
        patches = get_top_patches_all_features(
            layer,
            activations_by_layer[layer],
            index["paths"],
            batch_size=64,
        )
        with gzip.open(pkl_path, "wb") as _f:
            pickle.dump(patches, _f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  Saved → {pkl_path}")

    # Remap image_path values when pkl was built on a different machine.
    for _plist in patches.values():
        for _p in _plist:
            _stored = Path(_p["image_path"])
            if not _stored.exists():
                _local = _name_to_local.get(_stored.name)
                if _local:
                    _p["image_path"] = _local

    all_top_patches_by_layer[layer] = patches

# Primary-layer alias for visual inspection / gallery cells
all_top_patches = all_top_patches_by_layer[cfg.sae.primary_layer]
print(f"\nall_top_patches (primary layer {cfg.sae.primary_layer}): {len(all_top_patches):,} features")

## 4. CLIP auto-labeling — all layers

In [ ]:
"""CLIP auto-labeling — top-10 concept labels per feature, for every target layer.

Two-step flow for speed:
  Step 4a  Precompute patch embeddings in memory (CLIP-encodes the context crops).
  Step 4b  Label all 49k features via embedding lookups — seconds per run.
           Delete clip_labels_layer{N}_full.json to force relabeling.

Storing top-10 (not top-3) so that tier_promoted_label (section 6) can promote
a Tier 1/2 descriptor if it appears anywhere in the ranked list, without re-running
the embedding pipeline. The raw JSON always stores the full top-10; display cells
use the promoted label for annotation.

patch_embeddings_by_layer and feature_labels_by_layer are populated for all
target layers. Primary-layer aliases (patch_embeddings, feature_labels) are set
for the visual inspection and gallery cells below.
"""
import importlib, src.features, utils.clip_vocab
importlib.reload(src.features)
importlib.reload(utils.clip_vocab)
from src.features import load_clip_labeler, precompute_patch_embeddings, label_features_clip
from utils.clip_vocab import get_vocab
import json

clip_model, processor = load_clip_labeler()
vocab = get_vocab()
print(f"Vocab: {len(vocab):,} concepts")

patch_embeddings_by_layer = {}
feature_labels_by_layer   = {}

for layer in cfg.sae.target_layers:
    print(f"\n── Layer {layer} ──")
    patches = all_top_patches_by_layer[layer]

    # Step 4a — patch embeddings (always computed in-memory; not persisted to disk)
    embs = precompute_patch_embeddings(patches, clip_model, processor, batch_size=256)
    patch_embeddings_by_layer[layer] = embs

    # Step 4b — CLIP labels (persisted; delete json to force refresh)
    labels_cache = feature_output_dir / f"clip_labels_layer{layer}_full.json"
    if labels_cache.exists():
        with open(labels_cache, encoding="utf-8") as _f:
            labels = {int(k): v for k, v in json.load(_f).items()}
        print(f"  [cached] {len(labels):,} features — {labels_cache.name}")
    else:
        labels = label_features_clip(
            patches, vocab, clip_model, processor,
            top_n=10, patch_embeddings=embs,
        )
        with open(labels_cache, "w", encoding="utf-8") as _f:
            json.dump({str(k): v for k, v in labels.items()}, _f, indent=2)
        print(f"  Saved → {labels_cache}")

    feature_labels_by_layer[layer] = labels
    for fi in list(labels)[:3]:
        print(f"  feature {fi:5d}: {labels[fi]}")

# Primary-layer aliases for visual inspection / gallery cells
patch_embeddings = patch_embeddings_by_layer[cfg.sae.primary_layer]
feature_labels   = feature_labels_by_layer[cfg.sae.primary_layer]
print(f"\nfeature_labels (primary layer {cfg.sae.primary_layer}): {len(feature_labels):,} features")

## 5. Monosemanticity Scores — all layers

In [ ]:
"""Visual inspection — features filtered by max activation, sorted by median.

Only features with max_activation >= cfg.features.min_activation_threshold are shown.
From the activation distribution (cell above):
  max < 1.0  → ~37k features — weak signal, labels dominated by noise
  max >= 1.0 → ~12k features — clear activations, reliable labels
Sorted by median activation so stable features surface above one-off outliers.
"""
import statistics
from src.visualise import make_patch_grid

N_INSPECT         = 20
MIN_ACT_THRESHOLD = cfg.features.min_activation_threshold

eligible = {
    fi: patches
    for fi, patches in all_top_patches.items()
    if patches and patches[0]["activation_value"] >= MIN_ACT_THRESHOLD
}
print(f"Features above threshold {MIN_ACT_THRESHOLD}: {len(eligible):,} / {len(all_top_patches):,}")

top_features = sorted(
    eligible,
    key=lambda fi: statistics.median(p["activation_value"] for p in eligible[fi]),
    reverse=True,
)[:N_INSPECT]

inspect_patches = {fi: all_top_patches[fi] for fi in top_features}
grid = make_patch_grid(inspect_patches, labels=feature_labels, context_patches=2, crop_size=128)
display(grid)

In [ ]:
"""Monosemanticity Score — compute for all features in every target layer.

Formula (Pach et al. 2025, Eq. 9):
  MS^k = Σ_{n<m} (ã^k_n · ã^k_m · s_{nm}) / Σ_{n<m} (ã^k_n · ã^k_m)
  s_{nm}  = CLIP cosine similarity between crop embeddings of patches n and m
  ã^k_n   = activation of feature k for patch n, min-max normalised to [0,1]

Uses cfg.features.ms_max_patches (top-5) not top-20: gives a wider, more
discriminating distribution because lower-ranked patches dilute the signal.
Requires patch_embeddings_by_layer from cell 4 — run that cell first.
"""
import importlib, src.features, src.visualise
importlib.reload(src.features)
importlib.reload(src.visualise)
from src.features import compute_monosemanticity_score
from src.visualise import plot_monosemanticity_distribution
import json, math

MS_MAX_PATCHES = cfg.features.ms_max_patches

figures_dir = repo_root / cfg.outputs.report_figures_path
figures_dir.mkdir(parents=True, exist_ok=True)

ms_scores_by_layer = {}

for layer in cfg.sae.target_layers:
    scores_cache = feature_output_dir / f"ms_scores_layer{layer}_top{MS_MAX_PATCHES}.json"

    if scores_cache.exists():
        with open(scores_cache, encoding="utf-8") as _f:
            scores = {int(k): v for k, v in json.load(_f).items()}
        print(f"[cached] layer {layer}: {scores_cache.name}")
    else:
        scores = compute_monosemanticity_score(
            all_top_patches_by_layer[layer],
            patch_embeddings_by_layer[layer],
            max_patches=MS_MAX_PATCHES,
        )
        with open(scores_cache, "w", encoding="utf-8") as _f:
            json.dump({str(k): v for k, v in scores.items()}, _f)
        print(f"Saved → {scores_cache}")

    ms_scores_by_layer[layer] = scores

    alive = [v for v in scores.values() if not math.isnan(v)]
    dead  = sum(1 for v in scores.values() if math.isnan(v))
    print(f"  layer {layer}: {len(alive):,} active, {dead:,} dead"
          f" | median={sorted(alive)[len(alive)//2]:.3f}"
          f" | 80th={sorted(alive)[int(len(alive)*0.8)]:.3f}")

    fig = plot_monosemanticity_distribution(
        scores,
        layer=layer,
        save_path=str(figures_dir / f"ms_dist_layer{layer}_top{MS_MAX_PATCHES}.png"),
    )
    display(fig)

# Primary-layer alias for the gallery cell below
ms_scores = ms_scores_by_layer[cfg.sae.primary_layer]
print(f"\nms_scores (primary layer {cfg.sae.primary_layer}): {len(ms_scores):,} features")

## 6. Feature gallery — top 50 by score

In [ ]:
"""Tier-promotion helper — display-time only, no re-scoring.

feature_labels stores top-10 cosine-ranked candidates per feature.
tier_promoted_label walks that list and returns the first Tier 1/2 hit
so that descriptive texture/part labels surface above ImageNet class names
when CLIP ranks them anywhere in the top-10.

The raw feature_labels values (unconstrained ranking) are unchanged in the
JSON cache and used for MS scoring; promoted labels are display/annotation only.
"""
from utils.clip_vocab import TIER1, TIER2

_TIER1_SET = {s.lower() for s in TIER1}
_TIER2_SET = {s.lower() for s in TIER2}

def tier_promoted_label(labels: list[str]) -> tuple[str, str]:
    """Return (promoted_label, source_tier) from a top-10 label list.
    Promotes the first Tier 1/2 hit if it appears anywhere in the list.
    Falls back to top-1 overall. Cosine ranking is unchanged."""
    for label in labels:
        if label.lower() in _TIER1_SET:
            return label, "T1"
        if label.lower() in _TIER2_SET:
            return label, "T2"
    if labels:
        return labels[0], "T4"
    return "?", "T4"

# Build promoted_labels: {feat_idx: [promoted_label, ...raw_top2]}
# First entry is the promoted label; rest are the next two raw candidates
# (excluding the promoted one) so make_patch_grid still shows 3 strings per row.
promoted_labels: dict[int, list[str]] = {}
tier_sources: dict[int, str] = {}
for fi, lbls in feature_labels.items():
    if not lbls:
        promoted_labels[fi] = []
        tier_sources[fi] = "?"
        continue
    best, tier = tier_promoted_label(lbls)
    tier_sources[fi] = tier
    raw_top2 = [l for l in lbls[:3] if l != best][:2]
    promoted_labels[fi] = [best] + raw_top2

tier_counts = {}
for t in tier_sources.values():
    tier_counts[t] = tier_counts.get(t, 0) + 1
print("Tier promotion summary (primary layer):", tier_counts)

In [ ]:
"""Feature gallery — top 50 features by Monosemanticity Score (primary layer).

We do NOT use sorted(ms_scores)[:50] directly: the MS distribution has a spike
at exactly 1.0 dominated by low-support features (2-3 near-identical patches) —
a small-sample artefact of the Pach estimator, not true monosemanticity, and a
naive top-k pulls those straight to the top. So we filter first — requiring the
full ms_max_patches support and a real max activation — then rank survivors by
MS. This surfaces the right-shoulder (~0.65-0.90) features: high similarity
sustained across 5 distinct high-activation patches = the cleanest, most
nameable concepts, which feed the manual annotation in section 7.

Labels shown are tier-promoted: the first Tier 1/2 descriptor that appears
anywhere in each feature's top-10 cosine list. Falls back to top-1 (Tier 4)
when no Tier 1/2 concept ranks in the top-10. Tier source is appended so the
annotation step can distinguish promoted vs. unconstrained labels.
"""
import importlib, src.visualise
importlib.reload(src.visualise)
from src.visualise import plot_feature_gallery
import math

MIN_ACTIVATION = cfg.features.min_activation_threshold
MIN_PATCHES    = cfg.features.ms_max_patches
TOP_N          = cfg.features.monosemanticity_top_n

# Support filter → drops the MS=1.0 artefact spike + weak-activation noise
eligible = []
for fi, score in ms_scores.items():
    if score is None or math.isnan(score):
        continue
    valid = [p for p in all_top_patches.get(fi, []) if p.get("patch_row") is not None]
    if len(valid) < MIN_PATCHES:
        continue
    if max((p["activation_value"] for p in valid), default=0.0) < MIN_ACTIVATION:
        continue
    eligible.append((fi, score))

eligible.sort(key=lambda kv: kv[1], reverse=True)
top_50 = [fi for fi, _ in eligible[:TOP_N]]

print(f"Support filter (>= {MIN_PATCHES} patches, max_act >= {MIN_ACTIVATION}): "
      f"{len(eligible):,} eligible → selected top {len(top_50)} by MS")
if top_50:
    print(f"MS range of selection: {ms_scores[top_50[0]]:.3f} (top) "
          f"… {ms_scores[top_50[-1]]:.3f} (#{len(top_50)})")

# Use tier-promoted labels for gallery display; fall back to raw feature_labels if
# the promotion cell hasn't been run yet (e.g. running cells out of order).
if "promoted_labels" in dir() and "tier_sources" in dir():
    _labels = {
        fi: promoted_labels.get(fi, []) + [f"[{tier_sources.get(fi,'?')}]"]
        for fi in top_50
    }
else:
    _labels = feature_labels if "feature_labels" in dir() else None

gallery_path = figures_dir / f"feature_gallery_layer{cfg.sae.primary_layer}.png"
fig = plot_feature_gallery(
    all_top_patches, top_50,
    labels=_labels,
    scores=ms_scores,
    max_patches=MIN_PATCHES,
    save_path=str(gallery_path),
)
display(fig)
print(f"Saved → {gallery_path}")

## 7. Manual annotation

For each top-50 feature assign a category:
**texture / color / part / scene / semantic / unclear**

Save to `report/notes/feature_catalog_layer9.md`.

## Checkpoint
- [ ] `top_patches_layer{4,6,9}_full.pkl.gz` saved to `outputs/features/`
- [ ] `clip_labels_layer{4,6,9}_full.json` saved to `outputs/features/`
- [ ] `ms_scores_layer{4,6,9}_top5.json` saved to `outputs/features/`
- [ ] MS distribution plotted for each layer
- [ ] `feature_gallery_layer9.png` saved to `report/figures/`
- [ ] Top 50 features (primary layer) manually annotated